In [14]:
from urllib.request import urlopen
import pandas as pd
import json

response = urlopen("https://api.openf1.org/v1/drivers?&session_key=latest")
data = json.loads(response.read().decode('utf-8'))
pd.DataFrame(data)

response = urlopen("https://api.openf1.org/v1/sessions?year=2026")
data = json.loads(response.read().decode('utf-8'))
pd.DataFrame(data)

,session_key,session_type,session_name,date_start,date_end,meeting_key,circuit_key,circuit_short_name,country_key,country_code,country_name,location,gmt_offset,year
0,11465,Practice,Day 1,2026-02-11T07:00:00+00:00,2026-02-11T16:00:00+00:00,1304,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
1,11466,Practice,Day 2,2026-02-12T07:00:00+00:00,2026-02-12T16:00:00+00:00,1304,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
2,11467,Practice,Day 3,2026-02-13T07:00:00+00:00,2026-02-13T16:00:00+00:00,1304,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
3,11470,Practice,Day 1,2026-02-18T07:00:00+00:00,2026-02-18T16:00:00+00:00,1305,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
4,11469,Practice,Day 2,2026-02-19T07:00:00+00:00,2026-02-19T16:00:00+00:00,1305,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,11429,Practice,Practice 1,2026-12-04T09:30:00+00:00,2026-12-04T10:30:00+00:00,1302,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Marina,04:00:00,2026
122,11430,Practice,Practice 2,2026-12-04T13:00:00+00:00,2026-12-04T14:00:00+00:00,1302,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Marina,04:00:00,2026
123,11431,Practice,Practice 3,2026-12-05T10:30:00+00:00,2026-12-05T11:30:00+00:00,1302,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Marina,04:00:00,2026
124,11432,Qualifying,Qualifying,2026-12-05T14:00:00+00:00,2026-12-05T15:00:00+00:00,1302,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Marina,04:00:00,2026


In [15]:
def get_session_data(year = 2026):
    response = urlopen(f"https://api.openf1.org/v1/sessions?year={year}")
    data = json.loads(response.read().decode('utf-8'))
    return pd.DataFrame(data)

def get_driver_data(session_key = "latest"):
    response = urlopen(f"https://api.openf1.org/v1/drivers?&session_key={session_key}")
    data = json.loads(response.read().decode('utf-8'))
    return pd.DataFrame(data)

def get_session_results(session_key = "latest"):
    response = urlopen(f"https://api.openf1.org/v1/session_result?session_key={session_key}")
    data = json.loads(response.read().decode('utf-8'))
    return pd.DataFrame(data)

def get_drivers_championship_standings(session_key = "latest"):
    response = urlopen(f"https://api.openf1.org/v1/championship_drivers?session_key={session_key}")
    data = json.loads(response.read().decode('utf-8'))
    return pd.DataFrame(data)

In [16]:
def get_top_k_position_scores(session_key = "latest", year = 2026, k = 10):
    # Get API data
    before_race_drivers_championship_standings = get_drivers_championship_standings(session_key)[["driver_number", "position_start"]]
    drivers = get_driver_data(session_key)
    session_results = get_session_results(session_key)[["driver_number", "position"]]

    # Merge dataframes to get the final output
    race_output = session_results.merge(
        drivers[["driver_number", "name_acronym"]], 
        left_on="driver_number", 
        right_on="driver_number", 
        how="left"
    ).merge(
        before_race_drivers_championship_standings[["driver_number", "position_start"]],
        on="driver_number",
        how="left"
    )
    return race_output[race_output["position"] <= k].sort_values("position")

In [17]:
session = get_session_data(year = 2026)

In [18]:
session

,session_key,session_type,session_name,date_start,date_end,meeting_key,circuit_key,circuit_short_name,country_key,country_code,country_name,location,gmt_offset,year
0,11465,Practice,Day 1,2026-02-11T07:00:00+00:00,2026-02-11T16:00:00+00:00,1304,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
1,11466,Practice,Day 2,2026-02-12T07:00:00+00:00,2026-02-12T16:00:00+00:00,1304,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
2,11467,Practice,Day 3,2026-02-13T07:00:00+00:00,2026-02-13T16:00:00+00:00,1304,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
3,11470,Practice,Day 1,2026-02-18T07:00:00+00:00,2026-02-18T16:00:00+00:00,1305,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
4,11469,Practice,Day 2,2026-02-19T07:00:00+00:00,2026-02-19T16:00:00+00:00,1305,63,Sakhir,36,BRN,Bahrain,Bahrain,03:00:00,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,11429,Practice,Practice 1,2026-12-04T09:30:00+00:00,2026-12-04T10:30:00+00:00,1302,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Marina,04:00:00,2026
122,11430,Practice,Practice 2,2026-12-04T13:00:00+00:00,2026-12-04T14:00:00+00:00,1302,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Marina,04:00:00,2026
123,11431,Practice,Practice 3,2026-12-05T10:30:00+00:00,2026-12-05T11:30:00+00:00,1302,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Marina,04:00:00,2026
124,11432,Qualifying,Qualifying,2026-12-05T14:00:00+00:00,2026-12-05T15:00:00+00:00,1302,70,Yas Marina Circuit,21,UAE,United Arab Emirates,Yas Marina,04:00:00,2026


In [39]:
session = session.sort_values("date_start")
session = session[(session.session_name == "Race") | (session.session_name == "Sprint")]

In [46]:
round_number_key = session[(session.session_name == "Race")].reset_index(drop=True).reset_index(names="round_number")[["meeting_key","round_number"]]
round_number_key["round_number"] = round_number_key["round_number"] + 1

In [47]:
round_number_key

,meeting_key,round_number
0,1279,1
1,1280,2
2,1281,3
3,1282,4
4,1283,5
5,1284,6
6,1285,7
7,1286,8
8,1287,9
9,1288,10


In [ ]:
from leaderboard import ResultAggregator

result_aggregator = ResultAggregator(year = 2025, round_number=1)


Fetching data from: https://api.openf1.org/v1/sessions?year=2025
